# sentimentkg-analysis
## Analysis Pipeline for Sentiment Knowledge Graph

This notebook performs exploratory and statistical analysis of sentiment and network data
extracted for the sentiment knowledge graph project.

**Three analytical dimensions are addressed:**

1. **Sentiment distribution** — polarity scores and sentiment type frequencies across
   self-expressed (quote) and perceived (description) text sources
2. **Sentiment coherence** — alignment between internalized and externalized sentiment
   using cosine similarity
3. **Activity network** — one-mode and two-mode network construction, centrality metrics,
   and sentiment-network integration for the most central nodes

**Inputs:** `text_aspec_senti_desc.csv`, `text_aspec_senti_quotes.csv`, `rolescomps_num.csv`  
**Outputs:** `sentiaspec_paps_desc.csv`, `sentiaspec_paps_quot.csv`, `total_sentiaspec_paps.csv`,
`total_sentiaspec_paps_cosine.csv`, `netsenti_centralcluster_nodesco.csv`

## Environment Setup

Run this notebook under the `base` conda environment (Python 3.12.8).

> The sentiment preprocessing step in `sentimentkg_build` requires the
> `Aspect-Based-Sentiment-Analysis` environment. This analysis notebook only requires
> standard scientific Python libraries.

```bash
conda activate base
```

---
## Part 1: Sentiment Distribution

### 1.1 Compute Proportional Polarity Score (PAPS)

The **Proportional Aspect Polarity Score (PAPS)** measures the average signed sentiment
per unit of text length, normalising for verbosity differences between graduates.

- Each aspect sentiment label (`positive`, `negative`, `neutral`) is mapped to `+1`, `−1`, `0`.
- The mean signed value across all extracted aspects is divided by the word count of the source text.
- A `Senti_Type` label is assigned based on the relative counts of sentiment classes.

In [ ]:
import pandas as pd
import ast


def analyze_sentiment_distribution(sentiments_str, text):
    """
    Compute the Proportional Aspect Polarity Score (PAPS) and classify the sentiment type
    for a single row.

    Parameters
    ----------
    sentiments_str : str
        Stringified dict mapping aspect strings to sentiment labels
        (e.g. "{'food': 'positive', 'service': 'negative'}").
    text : str or float
        Source text used to normalise the polarity score by word count.

    Returns
    -------
    pd.Series
        - ``AvgProp_Polarity`` (float): PAPS value rounded to 5 decimal places.
        - ``Senti_Type`` (str): one of 'mostly positive', 'mostly negative',
          'mixed', 'mostly neutral', 'uncertain', or 'none'.
    """
    try:
        sentiments = ast.literal_eval(sentiments_str)
    except Exception:
        return pd.Series({"AvgProp_Polarity": 0, "Senti_Type": "invalid"})

    sentiment_map = {'positive': 1, 'negative': -1, 'neutral': 0}
    sentiment_values = [sentiment_map.get(v, 0) for v in sentiments.values()]
    num_aspects = len(sentiment_values)

    text = str(text) if text is not None else ""
    text_length = len(text.split())

    if num_aspects == 0 or text_length == 0:
        return pd.Series({"AvgProp_Polarity": 0, "Senti_Type": "none"})

    avg_sentiment             = sum(sentiment_values) / num_aspects
    avg_proportional_polarity = avg_sentiment / text_length

    num_pos = sentiment_values.count(1)
    num_neg = sentiment_values.count(-1)
    num_neu = sentiment_values.count(0)

    if num_neu / num_aspects >= 0.5:
        sentiment_type = "mostly neutral"
    elif num_pos > 0 and num_neg > 0:
        sentiment_type = "mixed"
    elif num_pos > num_neg:
        sentiment_type = "mostly positive"
    elif num_neg > num_pos:
        sentiment_type = "mostly negative"
    else:
        sentiment_type = "uncertain"

    return pd.Series({
        "AvgProp_Polarity": round(avg_proportional_polarity, 5),
        "Senti_Type":       sentiment_type,
    })

### 1.2 Apply PAPS to Descriptions and Quotes

In [ ]:
# Descriptions — perceived sentiment
df_desc = pd.read_csv("processed_dat/text_dat/text_aspec_senti_desc.csv")
df_desc[['AvgProp_Polarity', 'Senti_Type']] = df_desc.apply(
    lambda row: analyze_sentiment_distribution(row['Sentiments'], row['Description']),
    axis=1,
)
df_desc.to_csv("processed_dat/senti_dat/sentiaspec_paps_desc.csv", index=False)
print(f"Saved: sentiaspec_paps_desc.csv  ({len(df_desc)} rows)")
df_desc.head()

In [ ]:
# Quotes — self-expressed sentiment
df_quot = pd.read_csv("processed_dat/text_dat/text_aspec_senti_quotes.csv")
df_quot[['AvgProp_Polarity', 'Senti_Type']] = df_quot.apply(
    lambda row: analyze_sentiment_distribution(row['Sentiments'], row['Quote']),
    axis=1,
)
df_quot.to_csv("processed_dat/senti_dat/sentiaspec_paps_quot.csv", index=False)
print(f"Saved: sentiaspec_paps_quot.csv  ({len(df_quot)} rows)")
df_quot.head()

### 1.3 Sentiment Type Distribution

Bar charts showing how frequently each sentiment type appears across the two text sources.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, (csv_path, title, color) in zip(axes, [
    ("processed_dat/senti_dat/sentiaspec_paps_desc.csv", "Perceived (Description)", "#fbaed2"),
    ("processed_dat/senti_dat/sentiaspec_paps_quot.csv", "Self-Expressed (Quote)",  "#f77fbe"),
]):
    df = pd.read_csv(csv_path)
    df['Senti_Type'].value_counts().plot(kind='bar', color=color, edgecolor='black', ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Frequency")
    ax.tick_params(axis='x', rotation=25)

plt.suptitle("Distribution of Sentiment Types", fontsize=14)
plt.tight_layout()
plt.show()

### 1.4 Merge Descriptions and Quotes

Combine both sentiment DataFrames on `id`, prefixing columns to disambiguate sources.

In [ ]:
df_quot = pd.read_csv("processed_dat/senti_dat/sentiaspec_paps_quot.csv")
df_desc = pd.read_csv("processed_dat/senti_dat/sentiaspec_paps_desc.csv")

df_quot = df_quot.rename(columns={
    'Aspects':         'quot_Aspects',
    'Sentiments':      'quot_Sentiments',
    'AvgProp_Polarity':'quot_AvgProp_Polarity',
    'Senti_Type':      'quot_Senti_Type',
})

df_desc = df_desc.rename(columns={
    'Aspects':         'desc_Aspects',
    'Sentiments':      'desc_Sentiments',
    'AvgProp_Polarity':'desc_AvgProp_Polarity',
    'Senti_Type':      'desc_Senti_Type',
})

merged_df = pd.merge(df_quot, df_desc, on='id', how='inner')
merged_df.to_csv("processed_dat/senti_dat/total_sentiaspec_paps.csv", index=False)
print(f"Saved: total_sentiaspec_paps.csv  ({len(merged_df)} rows)")

# Verify PAPS ranges
for col in ['desc_AvgProp_Polarity', 'quot_AvgProp_Polarity']:
    print(f"{col}: min={merged_df[col].min():.5f}, max={merged_df[col].max():.5f}")

### 1.5 PAPS Distribution — Violin + Box Plots

#### Per Sentiment Type (side-by-side)

Paired violin plots for each sentiment type reveal how PAPS values differ between
perceived and self-expressed text within the same emotional category.

In [ ]:
import numpy as np
from matplotlib.patches import Patch

onemode_df = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps.csv")
onemode_df = onemode_df[
    (onemode_df['quot_Senti_Type'] != 'none') &
    (onemode_df['desc_Senti_Type'] != 'none')
]

sentiment_types = sorted(
    set(onemode_df['desc_Senti_Type'].unique()) |
    set(onemode_df['quot_Senti_Type'].unique())
)

desc_data, quot_data = [], []
all_means, all_medians = [], []

for stype in sentiment_types:
    d = onemode_df[onemode_df['desc_Senti_Type'] == stype]['desc_AvgProp_Polarity']
    q = onemode_df[onemode_df['quot_Senti_Type'] == stype]['quot_AvgProp_Polarity']
    desc_data.append(d)
    quot_data.append(q)
    all_means.append((d.mean() if not d.empty else np.nan, q.mean() if not q.empty else np.nan))
    all_medians.append((d.median() if not d.empty else np.nan, q.median() if not q.empty else np.nan))

positions = np.arange(1, len(sentiment_types) + 1)
offset    = 0.2

plt.figure(figsize=(14, 7))

# Violin plots
parts_desc = plt.violinplot(desc_data, positions=positions - offset,
                            vert=False, showmeans=True, widths=0.40)
parts_quot = plt.violinplot(quot_data, positions=positions + offset,
                            vert=False, showmeans=True, widths=0.40)

for body in parts_desc['bodies']:
    body.set_facecolor('pink');   body.set_edgecolor('black'); body.set_alpha(0.8)
for body in parts_quot['bodies']:
    body.set_facecolor('#ffe135'); body.set_edgecolor('black'); body.set_alpha(0.8)

# Box plot overlays
box_kwargs_desc = dict(patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor='#dda0dd', color='#f400a1', linewidth=1.5, alpha=0.5),
                       medianprops=dict(color='#f400a1', linewidth=2),
                       whiskerprops=dict(color='#f400a1', linewidth=3),
                       capprops=dict(color='#f400a1', linewidth=2))
box_kwargs_quot = dict(patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor='#dda0dd', color='#bf00ff', linewidth=1.5, alpha=0.4),
                       medianprops=dict(color='#bf00ff', linewidth=2),
                       whiskerprops=dict(color='#bf00ff', linewidth=3),
                       capprops=dict(color='#bf00ff', linewidth=2))

for pos, d, q in zip(positions, desc_data, quot_data):
    plt.boxplot(d, positions=[pos - offset], vert=False, widths=0.30, **box_kwargs_desc)
    plt.boxplot(q, positions=[pos + offset], vert=False, widths=0.30, **box_kwargs_quot)

# Annotations
for pos, (dm, qm), (dmed, qmed) in zip(positions, all_means, all_medians):
    if not np.isnan(dm):
        plt.text(dm   + 0.35, pos - offset + 0.07, f'Mean: {dm:.3f}',   ha='right', color='#1f77b4', fontsize=10)
    if not np.isnan(dmed):
        plt.text(dmed + 0.35, pos - offset - 0.07, f'Median: {dmed:.3f}', ha='right', color='#f400a1', fontsize=10)
    if not np.isnan(qm):
        plt.text(qm   + 0.45, pos + offset + 0.07, f'Mean: {qm:.3f}',   ha='right', color='#ff7f00', fontsize=10)
    if not np.isnan(qmed):
        plt.text(qmed + 0.45, pos + offset - 0.07, f'Median: {qmed:.3f}', ha='right', color='#bf00ff', fontsize=10)

plt.yticks(positions, sentiment_types, rotation=90)
x_min = min(onemode_df['quot_AvgProp_Polarity'].min(), onemode_df['desc_AvgProp_Polarity'].min())
x_max = max(onemode_df['quot_AvgProp_Polarity'].max(), onemode_df['desc_AvgProp_Polarity'].max())
plt.xticks(np.arange(np.floor(x_min), np.ceil(x_max) + 0.05, 0.05))
plt.xlim(x_min, x_max)
plt.xlabel('PAPS Values')
plt.axvline(0, color='red', linestyle='dashed', linewidth=1.5)
plt.grid(True, color='lightgray', linestyle='--', linewidth=0.5)
plt.legend(handles=[
    Patch(facecolor='#ffe135', edgecolor='black', label='Self-Expressed (Quote)'),
    Patch(facecolor='pink',    edgecolor='black', label='Perceived (Description)'),
], loc='upper right')
plt.title('Distribution of PAPS Per Sentiment Type')
plt.tight_layout()
plt.savefig('saved figures/whiskerviolin_quotdesc_papstype.png', dpi=300, bbox_inches='tight')
plt.show()

#### Overall PAPS Distribution (horizontal violin + box)

In [ ]:
from matplotlib.lines import Line2D

onemode_df = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps.csv")

fig, ax = plt.subplots(figsize=(15, 6))

data   = [onemode_df['quot_AvgProp_Polarity'], onemode_df['desc_AvgProp_Polarity']]
labels = ['Quote', 'Description']
colors = ['#ffdf00', 'pink']

parts = ax.violinplot(data, showmeans=True, widths=0.95, vert=False)
for i, part in enumerate(parts['bodies']):
    part.set_facecolor(colors[i]); part.set_edgecolor('black'); part.set_alpha(0.8)

for i, values in enumerate(data):
    ax.boxplot(values, positions=[i + 1], widths=0.5, vert=False, patch_artist=True,
               showfliers=False,
               boxprops=dict(facecolor='#dda0dd', color='#bf00ff', linewidth=1.5, alpha=0.5),
               medianprops=dict(color='#bf00ff', linewidth=2),
               whiskerprops=dict(color='#bf00ff', linewidth=3),
               capprops=dict(color='#bf00ff', linewidth=2))
    median_val = np.median(values)
    mean_val   = np.mean(values)
    ax.annotate(f'Median: {median_val:.2f}',
                xy=(median_val, i + 1), xytext=(median_val + 0.20, i + 1 + 0.10),
                color='#bf00ff', fontsize=11)
    ax.annotate(f'Mean: {mean_val:.2f}',
                xy=(mean_val, i + 1), xytext=(mean_val + 0.15, i + 1 + 0.25),
                color='#1f77b4', fontsize=11)

ax.set_yticks([1, 2])
ax.set_yticklabels(labels, rotation=90)

y_min = min(onemode_df['quot_AvgProp_Polarity'].min(), onemode_df['desc_AvgProp_Polarity'].min())
y_max = max(onemode_df['quot_AvgProp_Polarity'].max(), onemode_df['desc_AvgProp_Polarity'].max())
ax.set_xticks(np.arange(np.floor(y_min), np.ceil(y_max) + 0.05, 0.05))
ax.set_xlim(y_min, y_max)
ax.set_xlabel('PAPS Values')
ax.set_title('Distribution of Positive to Negative Sentiments')
ax.axvline(0, color='red', linestyle='dashed', linewidth=1.5)
ax.grid(True, color='lightgray', linestyle='--', linewidth=0.5)
ax.legend(handles=[
    Line2D([0], [0], color='#f5c71a', lw=4, label='Quote (Self-Expressed)'),
    Line2D([0], [0], color='#f77fbe', lw=4, label='Description (Perceived)'),
], loc='upper right')

plt.tight_layout()
plt.savefig('saved figures/whiskerviolin_quotdesc_paps.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.6 PAPS Distribution — Histogram + KDE

In [ ]:
from scipy.stats import gaussian_kde

merged_df = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps.csv")

desc_data = merged_df['desc_AvgProp_Polarity'].dropna()
quot_data = merged_df['quot_AvgProp_Polarity'].dropna()

x_min = min(desc_data.min(), quot_data.min())
x_max = max(desc_data.max(), quot_data.max())
x_vals = np.linspace(x_min, x_max, 500)

plt.figure(figsize=(12, 6))

plt.hist(desc_data, bins=20, color='pink',    edgecolor='black', alpha=0.8,
         density=True, label='Description (Perceived)')
plt.hist(quot_data, bins=20, color='#ffdf00', edgecolor='black', alpha=0.6,
         density=True, label='Quote (Self-Expressed)')

desc_kde_vals = gaussian_kde(desc_data)(x_vals)
quot_kde_vals = gaussian_kde(quot_data)(x_vals)
plt.plot(x_vals, desc_kde_vals, color='#f400a1', linewidth=2, label='KDE – Description')
plt.plot(x_vals, quot_kde_vals, color='#ff7f00', linewidth=2, label='KDE – Quote')

for peak_x, peak_y, color in [
    (x_vals[np.argmax(desc_kde_vals)], np.max(desc_kde_vals), '#f400a1'),
    (x_vals[np.argmax(quot_kde_vals)], np.max(quot_kde_vals), '#ff7f00'),
]:
    plt.annotate(f'{peak_y:.2f}', xy=(peak_x, peak_y),
                 xytext=(peak_x + 0.10, peak_y + 0.05),
                 arrowprops=dict(arrowstyle='-', color=color, lw=1.5),
                 color=color, fontsize=12)

plt.xlim(x_min, x_max)
plt.axvline(0, color='red', linestyle='--', linewidth=1.5)
plt.xlabel('PAPS Values'); plt.ylabel('Density')
plt.title('Frequency of Positive to Negative Sentiments')
plt.legend()
plt.grid(True, color='lightgray', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig('saved figures/hist_quotdesc_paps.png', dpi=300, bbox_inches='tight')
plt.show()

### 1.7 PAPS Descriptive Statistics per Sentiment Type

In [ ]:
merged_df = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps.csv")
merged_df = merged_df[
    (merged_df['quot_Senti_Type'] != 'none') &
    (merged_df['desc_Senti_Type'] != 'none')
]

sentiment_types = sorted(
    set(merged_df['desc_Senti_Type'].unique()) |
    set(merged_df['quot_Senti_Type'].unique())
)

for stype in sentiment_types:
    for label, col_type, col_paps in [
        ('Perceived (Description)', 'desc_Senti_Type', 'desc_AvgProp_Polarity'),
        ('Self-Expressed (Quote)',  'quot_Senti_Type', 'quot_AvgProp_Polarity'),
    ]:
        data = merged_df[merged_df[col_type] == stype][col_paps].dropna()
        if data.empty:
            print(f"{label} — {stype}: No data")
            continue
        q1, q2, q3 = data.quantile([0.25, 0.5, 0.75])
        iqr  = q3 - q1
        print(f"{label} — {stype}:")
        print(f"  Q1: {q1:.4f}  |  Median: {q2:.4f}  |  Q3: {q3:.4f}  |  IQR: {iqr:.4f}")
        print(f"  Whisker range: [{data[data >= q1 - 1.5*iqr].min():.4f}, "
              f"{data[data <= q3 + 1.5*iqr].max():.4f}]")
        print(f"  True range:    [{data.min():.4f}, {data.max():.4f}]")
    print('-' * 60)

---
## Part 2: Sentiment Coherence

**Sentiment coherence** captures how closely a graduate's self-expressed sentiment (quote)
aligns with the sentiment others assigned to them (description).

Cosine similarity is used as the alignment metric because it normalises for vector length,
isolating directional agreement between the two sentiment profiles regardless of how many
aspects were extracted from each text.

### 2.1 Compute Cosine Similarity (Senti_Align)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import ast

merged_df = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps.csv")

sentiment_mapping = {'positive': 1, 'neutral': 0, 'negative': -1}


def parse_sentiment(sentiment):
    """
    Parse a stringified sentiment dict into a list of signed numeric values.

    Parameters
    ----------
    sentiment : str or list
        Either a raw string representation of a dict (e.g. "{'food': 'positive'}")
        or an already-parsed list.

    Returns
    -------
    list of int
        Signed values: +1 (positive), 0 (neutral), −1 (negative).
    """
    if isinstance(sentiment, str):
        try:
            parsed = ast.literal_eval(sentiment)
            return [sentiment_mapping.get(val, 0) for val in parsed.values()]
        except (ValueError, SyntaxError):
            return []
    return sentiment if isinstance(sentiment, list) else []


def compute_cosine_similarity(quot_sentiments, desc_sentiments):
    """
    Compute cosine similarity between two variable-length sentiment vectors.

    Vectors are zero-padded to the same length before normalisation. Returns NaN
    if either vector is empty or has zero norm.
    """
    if not quot_sentiments or not desc_sentiments:
        return np.nan

    max_len     = max(len(quot_sentiments), len(desc_sentiments))
    quot_vector = np.array(quot_sentiments + [0.0] * (max_len - len(quot_sentiments)))
    desc_vector = np.array(desc_sentiments + [0.0] * (max_len - len(desc_sentiments)))

    if np.linalg.norm(quot_vector) == 0 or np.linalg.norm(desc_vector) == 0:
        return np.nan

    quot_vector = quot_vector / np.linalg.norm(quot_vector)
    desc_vector = desc_vector / np.linalg.norm(desc_vector)

    return cosine_similarity(quot_vector.reshape(1, -1), desc_vector.reshape(1, -1))[0][0]


merged_df['quot_Sentiments'] = merged_df['quot_Sentiments'].apply(parse_sentiment)
merged_df['desc_Sentiments'] = merged_df['desc_Sentiments'].apply(parse_sentiment)

merged_df['Senti_Align'] = merged_df.apply(
    lambda row: compute_cosine_similarity(row['quot_Sentiments'], row['desc_Sentiments']),
    axis=1,
)

merged_df.to_csv("processed_dat/senti_dat/total_sentiaspec_paps_cosine.csv", index=False)
print(f"Saved: total_sentiaspec_paps_cosine.csv")
print(f"Senti_Align range: [{merged_df['Senti_Align'].min():.4f}, {merged_df['Senti_Align'].max():.4f}]")

### 2.2 Sentiment Coherence Distribution — Histogram + KDE

In [ ]:
from scipy.stats import gaussian_kde

merged_df = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps_cosine.csv")
data      = merged_df['Senti_Align'].dropna()

mean_val   = data.mean()
median_val = data.median()

plt.figure(figsize=(12, 6))
plt.hist(data, bins=20, color='#ca1f7b', edgecolor='black', alpha=0.8, density=True)

x_vals = np.linspace(-1, 1, 500)
plt.plot(x_vals, gaussian_kde(data)(x_vals), color='#f400a1', linewidth=2, label='KDE Curve')

plt.axvline(mean_val,   color='#ffcc33', linestyle='dashed', linewidth=2,
            label=f'Mean: {mean_val:.2f}')
plt.axvline(median_val, color='#30d5c8', linestyle='dashed', linewidth=2,
            label=f'Median: {median_val:.2f}')
plt.axvline(0, color='red', linestyle='--', linewidth=1.5)

plt.xlim(-1, 1)
plt.xlabel('Cosine Similarity'); plt.ylabel('Density')
plt.title('Distribution of Sentiment Alignment')
plt.legend()
plt.grid(True, color='lightgray', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.savefig('saved figures/hist_quotdesc_cosine.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Proportion of perfect alignment (cosine = 1.0): "
      f"{(data == 1.0).mean():.4f}")

### 2.3 Coherence Distribution per Sentiment Type

Side-by-side violin plots for each sentiment type, split by source (quote vs description),
show how alignment intensity varies across emotional categories.

In [ ]:
merged_df = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps_cosine.csv")
merged_df = merged_df.dropna(subset=['Senti_Align'])

sentiment_types = sorted(
    set(merged_df['desc_Senti_Type'].unique()) |
    set(merged_df['quot_Senti_Type'].unique())
)

desc_data, quot_data = [], []
all_means, all_medians = [], []

for stype in sentiment_types:
    d = merged_df[merged_df['desc_Senti_Type'] == stype]['Senti_Align']
    q = merged_df[merged_df['quot_Senti_Type'] == stype]['Senti_Align']
    desc_data.append(d);  quot_data.append(q)
    all_means.append((d.mean() if not d.empty else np.nan,
                      q.mean() if not q.empty else np.nan))
    all_medians.append((d.median() if not d.empty else np.nan,
                        q.median() if not q.empty else np.nan))

positions = np.arange(1, len(sentiment_types) + 1)
offset    = 0.2

plt.figure(figsize=(14, 6))

parts_desc = plt.violinplot(desc_data, positions=positions - offset, showmeans=True, widths=0.4)
parts_quot = plt.violinplot(quot_data, positions=positions + offset, showmeans=True, widths=0.4)

for body in parts_desc['bodies']:
    body.set_facecolor('pink');    body.set_edgecolor('black'); body.set_alpha(0.8)
for body in parts_quot['bodies']:
    body.set_facecolor('#ffe135'); body.set_edgecolor('black'); body.set_alpha(0.8)

box_kw_d = dict(patch_artist=True, showfliers=False,
                boxprops=dict(facecolor='#dda0dd', color='#f400a1', linewidth=1.5, alpha=0.5),
                medianprops=dict(color='#f400a1', linewidth=2),
                whiskerprops=dict(color='#f400a1', linewidth=3),
                capprops=dict(color='#f400a1', linewidth=2))
box_kw_q = dict(patch_artist=True, showfliers=False,
                boxprops=dict(facecolor='#dda0dd', color='#bf00ff', linewidth=1.5, alpha=0.5),
                medianprops=dict(color='#bf00ff', linewidth=2),
                whiskerprops=dict(color='#bf00ff', linewidth=3),
                capprops=dict(color='#bf00ff', linewidth=2))

for pos, d, q in zip(positions, desc_data, quot_data):
    plt.boxplot(d, positions=[pos - offset], widths=0.1, **box_kw_d)
    plt.boxplot(q, positions=[pos + offset], widths=0.1, **box_kw_q)

for pos, (dm, qm), (dmed, qmed) in zip(positions, all_means, all_medians):
    if not np.isnan(dm):
        plt.text(pos - offset - 0.25, dm - 0.02,   f'Mean: {dm:.3f}',
                 rotation=90, ha='center', color='#1f77b4', fontsize=10)
    if not np.isnan(dmed):
        plt.text(pos - offset - 0.15, dmed - 0.02, f'Median: {dmed:.3f}',
                 rotation=90, ha='center', color='#f400a1', fontsize=10)
    if not np.isnan(qm):
        plt.text(pos + offset + 0.25, qm + 0.02,   f'Mean: {qm:.3f}',
                 rotation=90, ha='center', color='#ff7f00', fontsize=10)
    if not np.isnan(qmed):
        plt.text(pos - offset + 0.55, qmed + 0.02, f'Median: {qmed:.3f}',
                 rotation=90, ha='center', color='#bf00ff', fontsize=10)

plt.xticks(positions, sentiment_types)
y_min, y_max = merged_df['Senti_Align'].min(), merged_df['Senti_Align'].max()
plt.yticks(np.arange(np.floor(y_min), np.ceil(y_max) + 0.10, 0.10))
plt.ylim(y_min, y_max)
plt.xlabel('Sentiment Types'); plt.ylabel('Alignment Intensity (Cosine Similarity)')
plt.title('Distribution of Sentiment Coherence Per Type')
plt.axhline(0, color='red', linestyle='dashed', linewidth=1.2)
plt.grid(True, color='lightgray', linestyle='--', linewidth=0.5)

from matplotlib.patches import Patch
plt.legend(handles=[
    Patch(facecolor='#ffe135', edgecolor='black', label='Self-Expressed (Quote)'),
    Patch(facecolor='pink',    edgecolor='black', label='Perceived (Description)'),
], loc='upper left')

plt.tight_layout()
plt.savefig('saved figures/whiskerviolin_quotdesc_cosine.png', dpi=300, bbox_inches='tight')
plt.show()

### 2.4 Coherence Descriptive Statistics per Sentiment Type

In [ ]:
onemode_df = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps_cosine.csv")
onemode_df = onemode_df.dropna(subset=['Senti_Align'])

sentiment_types = sorted(
    set(onemode_df['desc_Senti_Type'].unique()) |
    set(onemode_df['quot_Senti_Type'].unique())
)

for stype in sentiment_types:
    for label, col_type in [
        ('Perceived (Description)', 'desc_Senti_Type'),
        ('Self-Expressed (Quote)',  'quot_Senti_Type'),
    ]:
        data = onemode_df[onemode_df[col_type] == stype]['Senti_Align'].dropna()
        if data.empty:
            print(f"{label} — {stype}: No data"); continue
        q1, q2, q3 = data.quantile([0.25, 0.5, 0.75])
        iqr = q3 - q1
        print(f"{label} — {stype}:")
        print(f"  Q1: {q1:.4f}  |  Median: {q2:.4f}  |  Q3: {q3:.4f}  |  IQR: {iqr:.4f}")
        print(f"  Whisker range: [{data[data >= q1 - 1.5*iqr].min():.4f}, "
              f"{data[data <= q3 + 1.5*iqr].max():.4f}]")
        print(f"  True range:    [{data.min():.4f}, {data.max():.4f}]")
    print('-' * 60)

---
## Part 3: Activity Network

Graduate network data is represented as a one-mode co-membership graph (node-to-node)
and a two-mode bipartite affiliation graph (cadet-to-activity).

Network centrality measures used:

| Measure | Description |
|---|---|
| Degree centrality | Fraction of all other nodes a given node is directly connected to |
| Local clustering coefficient | Probability that a node's neighbours are also connected to each other |

## Environment Note

Switch to the `base` conda environment (Python 3.12.8) for `networkx` and `community`.

```bash
conda activate base
```

### 3.1 One-Mode Network — Co-Activity Graph

Each cadet is a node. An edge is drawn between two cadets who shared at least one activity,
weighted by the number of shared activities.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# Load data
onemode_df  = pd.read_csv('processed_dat/net_dat/rolescomps_num.csv')
activity_cols = [col for col in onemode_df.columns if col.startswith('activities.')]
group_cols    = ['Alfa', 'Bravo', 'Charlie', 'Delta', 'Echo', 'Foxtrot', 'Golf', 'Hawk']

activity_df = onemode_df[['id'] + activity_cols].copy()
for group in group_cols:
    activity_df[group] = onemode_df[group]

# Build graph
G = nx.Graph()

for _, row in activity_df.iterrows():
    group_affiliations = row[group_cols]
    primary_group = group_affiliations.idxmax() if group_affiliations.max() > 0 else 'No Group'
    G.add_node(row['id'], group=primary_group)

for i in range(len(activity_df)):
    for j in range(i + 1, len(activity_df)):
        shared = int(np.sum(
            (activity_df.iloc[i, 1:len(activity_cols) + 1] > 0) &
            (activity_df.iloc[j, 1:len(activity_cols) + 1] > 0)
        ))
        if shared > 0:
            G.add_edge(activity_df.iloc[i]['id'], activity_df.iloc[j]['id'], weight=shared)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Visualise one-mode graph
group_color_map = {
    'Alfa':    '#d12276', 'Bravo':   '#ff55a7', 'Charlie': '#bbd741',
    'Delta':   '#ff6503', 'Echo':    '#ffad31', 'Foxtrot': '#40e1cd',
    'Golf':    '#ea68ff', 'Hawk':    '#7f41ce', 'No Group':'#D3D3D3',
}

weight_color_map = {
    1: "#ffc0cb", 2: "#7fffd4", 3: "#ff9966",
    4: "#dda0dd", 5: "#d2ee59", 6: "#8b008b", 7: "#ff007f",
}

pos = nx.spring_layout(G, k=4, iterations=100, weight='weight', seed=42)
unique_groups = list(set(nx.get_node_attributes(G, 'group').values()))

plt.figure(figsize=(20, 12))

nx.draw_networkx_nodes(
    G, pos, node_size=1200, alpha=1,
    node_color=[group_color_map.get(G.nodes[n]['group'], '#cccccc') for n in G.nodes],
)

sorted_edges  = sorted(G.edges(data=True), key=lambda x: x[2]['weight'])
edge_colors   = [weight_color_map[e[2]['weight']] for e in sorted_edges]
edge_widths   = [(e[2]['weight'] / 7) * 4 + 1 for e in sorted_edges]

nx.draw_networkx_edges(
    G, pos, edgelist=[(u, v) for u, v, d in sorted_edges],
    edge_color=edge_colors, width=edge_widths, alpha=0.6,
)
nx.draw_networkx_labels(G, pos, font_size=11, font_color='white', font_weight='bold')

legend_companies = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=group_color_map.get(g, '#cccccc'),
           markersize=15, label=g)
    for g in sorted(unique_groups)
]
legend_weights = [
    Line2D([0], [0], color=c, lw=4, label=str(w))
    for w, c in sorted(weight_color_map.items())
]

l1 = plt.legend(handles=legend_companies, title="Company of Cadet", title_fontsize=16,
                 fontsize=14, loc='upper right', bbox_to_anchor=(1.15, 1), markerscale=1.5)
plt.legend(handles=legend_weights,   title="Number of Activities", title_fontsize=16,
           fontsize=14, loc='upper right', bbox_to_anchor=(1.15, 0.75), handlelength=3)
plt.gca().add_artist(l1)

plt.title("One-Mode Network: Shared Activity Among Cadets", fontsize=30)
plt.axis('off')
plt.tight_layout()
plt.savefig('saved figures/onemode_rolescomp_net.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.2 Network-Level Descriptive Statistics

In [ ]:
from community import community_louvain

print(f"Nodes:                        {G.number_of_nodes()}")
print(f"Edges:                        {G.number_of_edges()}")
print(f"Density:                      {nx.density(G):.4f}")
print(f"Average clustering coeff.:    {nx.average_clustering(G):.4f}")

degrees = [d for _, d in G.degree()]
print(f"Degree — mean: {np.mean(degrees):.4f}, median: {np.median(degrees):.4f}, "
      f"min: {min(degrees)}, max: {max(degrees)}")

avg_edges  = 2 * G.number_of_edges() / G.number_of_nodes()
avg_weight = sum(d['weight'] for _, _, d in G.edges(data=True)) / G.number_of_edges()
print(f"Avg. edges per node:          {avg_edges:.4f}")
print(f"Avg. weight per edge:         {avg_weight:.4f}")

partition       = community_louvain.best_partition(G, weight='weight')
num_communities = len(set(partition.values()))
print(f"Louvain communities:          {num_communities}")
for node in G.nodes():
    G.nodes[node]['community'] = partition[node]

#### Statistics per Company Sub-Graph

In [ ]:
group_dict = nx.get_node_attributes(G, 'group')

for group in sorted(set(group_dict.values())):
    group_nodes = [n for n, g in group_dict.items() if g == group]
    subG        = G.subgraph(group_nodes)

    print(f"\n--- {group} (n={subG.number_of_nodes()}) ---")
    if subG.number_of_nodes() > 1:
        print(f"  Edges:              {subG.number_of_edges()}")
        print(f"  Density:            {nx.density(subG):.4f}")
        print(f"  Avg. clustering:    {nx.average_clustering(subG):.4f}")
        degs = [d for _, d in subG.degree()]
        print(f"  Degree — mean: {np.mean(degs):.4f}, median: {np.median(degs):.4f}, "
              f"min: {min(degs)}, max: {max(degs)}")
        if subG.number_of_edges() > 0:
            avg_w = sum(d['weight'] for _, _, d in subG.edges(data=True)) / subG.number_of_edges()
            print(f"  Avg. weight/edge:  {avg_w:.4f}")
    else:
        print("  Insufficient nodes for sub-graph statistics.")

### 3.3 Most Central Nodes — Top 25%

Degree centrality is used to identify the top quartile of cadets by network position.
Local clustering coefficients and company affiliations are then appended to profile
each central node.

In [ ]:
import math

centrality = nx.degree_centrality(G)
top_n      = math.ceil(len(centrality) * 0.25)

top_25_percent = sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:top_n]
top_ids        = [node for node, _ in top_25_percent]

print(f"Top 25% nodes by degree centrality (n={top_n}):")
for node, cent in top_25_percent:
    print(f"  {node}: {cent:.4f}")

# Count isolated nodes
isolated = [n for n, d in G.degree() if d == 0]
print(f"\nIsolated nodes (degree 0): {len(isolated)}")

In [ ]:
# Build sentiment profile for top nodes
df_senti = pd.read_csv("processed_dat/senti_dat/total_sentiaspec_paps_cosine.csv")
selected_cols = ['id', 'quot_Sentiments', 'desc_Sentiments',
                 'quot_AvgProp_Polarity', 'desc_AvgProp_Polarity',
                 'quot_Senti_Type', 'desc_Senti_Type', 'Senti_Align']
df_senti[selected_cols].to_csv("processed_dat/senti_dat/senti_papscosinetype.csv", index=False)

# Subset to top central nodes
top_df = df_senti[df_senti['id'].isin(top_ids)][selected_cols].copy()

# Append degree centrality
centrality_df = pd.DataFrame(list(centrality.items()), columns=['id', 'Net_Centrality'])
top_df = top_df.merge(centrality_df, on='id', how='left')

# Append local clustering coefficients
clustering_coeffs = nx.clustering(G, nodes=top_ids)
top_df['Local_Clustering'] = top_df['id'].map(clustering_coeffs)

# Append company affiliations
co_df = pd.read_csv('processed_dat/basic_sword80.csv')
top_df = top_df.merge(co_df[['id', 'Company']], on='id', how='left')

top_df.to_csv('processed_dat/netsenti_centralcluster_nodesco.csv', index=False)
print(f"Saved: netsenti_centralcluster_nodesco.csv  ({len(top_df)} rows)")
top_df.head()

### 3.4 Centrality vs. Clustering — Scatter Plot

In [ ]:
df = pd.read_csv('processed_dat/netsenti_centralcluster_nodesco.csv')

plt.figure(figsize=(12, 8))
plt.scatter(df['Net_Centrality'], df['Local_Clustering'], color='blue', alpha=0.7)

for _, row in df.iterrows():
    plt.annotate(row['id'],
                 (row['Net_Centrality'], row['Local_Clustering']),
                 textcoords="offset points", xytext=(0, 5),
                 ha='center', fontsize=8)

plt.xlabel('Degree Centrality')
plt.ylabel('Local Clustering Coefficient')
plt.title('Degree Centrality vs. Local Clustering — Top 25% Nodes')
plt.grid(True)
plt.tight_layout()
plt.show()

### 3.5 Two-Mode Network — Cadet × Activity Bipartite Graph

Each cadet is a node linked to every activity they participated in. Edge weight encodes
their role level within that activity.

In [ ]:
onemode_df    = pd.read_csv('processed_dat/net_dat/rolescomps_num.csv')
activity_cols = [col for col in onemode_df.columns if col.startswith('activities.')]
group_cols    = ['Alfa', 'Bravo', 'Charlie', 'Delta', 'Echo', 'Foxtrot', 'Golf', 'Hawk']

B = nx.Graph()

for _, row in onemode_df.iterrows():
    cadet_id          = row['id']
    group_affiliations = row[group_cols]
    primary_group     = group_affiliations.idxmax() if group_affiliations.max() > 0 else 'No Group'
    B.add_node(cadet_id, bipartite=0, group=primary_group)

    for act in activity_cols:
        if row[act] > 0:
            B.add_node(act, bipartite=1, group='Activity')
            B.add_edge(cadet_id, act, weight=row[act])

print(f"Bipartite graph: {B.number_of_nodes()} nodes, {B.number_of_edges()} edges")

In [ ]:
pos = nx.spring_layout(B, k=3, iterations=100, weight='weight', seed=42)

node_colors = [group_color_map.get(B.nodes[n]['group'], '#aaaaaa') for n in B.nodes]
node_sizes  = [900 if B.nodes[n]['bipartite'] == 0 else 500 for n in B.nodes]

edge_weights = [d['weight'] for _, _, d in B.edges(data=True)]
max_weight   = max(edge_weights) if edge_weights else 1
edge_widths  = [(w / max_weight) * 4 + 1 for w in edge_weights]

unique_groups_B = sorted(set(nx.get_node_attributes(B, 'group').values()) - {'Activity'})

plt.figure(figsize=(20, 15))
nx.draw_networkx_nodes(B, pos, node_color=node_colors, node_size=node_sizes, alpha=1)
nx.draw_networkx_edges(B, pos, width=edge_widths, edge_color='#888', alpha=0.6)

cadet_nodes = [n for n, d in B.nodes(data=True) if d['bipartite'] == 0]
nx.draw_networkx_labels(B, pos, {n: n for n in cadet_nodes}, font_size=9, font_color='white')

legend_companies_B = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=group_color_map.get(g, '#cccccc'), markersize=12, label=g)
    for g in unique_groups_B
]
l1 = plt.legend(handles=legend_companies_B, title="Company", loc='upper right',
                bbox_to_anchor=(1.15, 1))
plt.gca().add_artist(l1)

plt.title("Two-Mode Network: Cadets and Their Activities", fontsize=20)
plt.axis('off')
plt.tight_layout()
plt.show()